# Notebook to accompany workshop 'From a mess to a map'

This workshop and notebook demonstrate some of the problems which may be encountered in preparing messy data so it can be examined using computational tools. The messy data we will work with is a set of responses to a question in an online survey - participants were asked to provide an Australian slang expression which they knew to mean 'Something very good'. The question of interest is whether there is variation by state in the proportion of the most popular responses. It is relatively easy to use computational tools to produce maps showing this sort of information, but the input to those tools has a particular, highly organised format.

We need to start out knowing what that final data format will look like. This is a very general rule: find out what data format is required by the tool you want to use before you start manipulating the data! In general, the mapping tool we will use (part of the ```ggplot2``` package in R) expects an input table where each geographic entity is a row and that row contains the geodata associated with the entity in one column and then any number of other columns representing attributes of the geographic entity which we are interested in. For the present exercise, this means a table with a row for each state of Australia which will contain the geodata and (minimally) the percentage response rate from that state for each of the five most popular responses.

Along the way, we will ignore some complications; I will try to at least acknowledge when this happens.

### Data Preparation:
Download Slang Survey data from https://data.ldaca.edu.au/collection?id=arcp%3A%2F%2Fname%2Chdl10.26180~30102115&_crateId=arcp%3A%2F%2Fname%2Chdl10.26180~30102115 and unzip to a suitable location, read the provenance file to understand how the data was collected.

We will work with two of the files:
- the file with responses to the first prompt, Item01-VeryGood.csv
- the file with information about the respondents, ParticipantDemographics.csv

### Code Preparation:
We will use six R packages:
- ```stringr```: functions for working with strings
- ```ozmaps```: geographic data for making maps of Australia
- ```ggplot2```: a widely used plotting package
- ```viridis```: colour palettes for plotting
- ```patchwork```: to create arrays of plots
- ```repr```: to handle setting options

If you don't have these packages installed already, you will need to uncomment (delete the *##* at the start of the lines) the ```install.packages``` lines in the following code cell.

In [ ]:
## install package if necessary
## install.packages("stringr")
## install.packages("ozmaps")
## install.packages("ggplot2")
## install.packages("viridis")
## install.packages("patchwork")
## install.packages("repr")

## load packages
library(stringr)
library(ozmaps)
library(ggplot2)
library(viridis)
library(patchwork)
library(repr)

# set plot options
options(repr.plot.width = 20, repr.plot.height = 12)

In [ ]:
## Load data files - fill in path to each file
responses <- read.csv('C:/Users/Simon/OneDrive/Desktop/slang/Item01-VeryGood.csv')
participants <- read.csv('C:/Users/Simon/OneDrive/Desktop/slang/ParticipantDemographics.csv')

In [ ]:
## let's see what we have
head(responses)

In [ ]:
head(participants)

The information which we need from the ```participants``` dataframe is the ID for each participant and their location when they completed the survey, ```custom.currentLocation```. First we extract this information to a new dataframe, then we get rid of any participants for whom we do not have a postcode. When you run the code in the second cell below, you will see a warning message ```NAs introduced by coercion``` - this is exactly what is intended. The function ```as.numeric``` tries to coerce values to be numbers and if there is text (or nothing) in the field, an NA (null) value is generated. This makes it easy to filter out the participants without postcodes. You will see that we go from 2984 participants to 2695 as a result of this filtering.

**Complication**: In the survey, we asked people whether in their childhood they lived in a different place to where they lived when they participated in the survey. The idea was that some aspects of language are fixed quite early in life and we might at some point be interested in tracking that in our data. There are two reasons why this may not be as useful as we thought:
- vocabulary is part of language which stays flexible through life (compared to e.g. pronunciation)
- geographic variation in the data seems to be more about proportions of choices rather than categorical choices and therefore it would be very hard to identify childhood influence. 

In [ ]:
## extract participantID, postcode
participants_postcodes <- participants[,c('participantID', 'custom.currentLocation')] 
colnames(participants_postcodes) <- c('participantID', 'postCode')
str(participants_postcodes)

In [ ]:
## clean up list
participants_postcodes$postCode <- as.numeric(participants_postcodes$postCode)
participants_postcodes <- participants_postcodes[!is.na(participants_postcodes$postCode), ]

In [ ]:
nrow(participants_postcodes)


Now we can import the postcodes into the table of response data using the ```merge``` function of R. We could specify that we want to match the two data sources on the ```participantID``` value, but as the two dataframes have only that column in common, it would be redundant. As you can see, the new dataframe has all the data from the response table plus a postcode for each participant.

In [ ]:
responses_postcodes <- merge(x = responses, y = participants_postcodes)

In [ ]:
head(responses_postcodes)

Now let's have a look at a listing of responses by frequency. If we look at the first few rows here, for example the entries for *bonza* and variants, we start to see the mess! But do please look further down. What other messy things do you see in the data?

In [ ]:
## look at frequency table for responses
sort(table(responses_postcodes$slang), decreasing = TRUE)

It is intuitively clear that some responses are variants of others. But the relationship is not the same in each case. We need to make decisions about how (or even whether) we want to group responses.

Starting with the easy case, we can see that e.g. *grouse* and *Grouse* are counted separately. Is there any reason why we would not normalise case for counting purposes? This is very easy to do: R has a function ```tolower```. Note that the code adds a new-column to the dataframe. This means that there is a provenance trail within the dataframe and this could be important if we decided to export this version of the data. The revised frequency table shows that this change has reduced mess a little.

In [ ]:
## all lower case
responses_postcodes$slang_lower <- tolower(responses_postcodes$slang)

In [ ]:
## look again
sort(table(responses_postcodes$slang_lower), decreasing = TRUE)

Two of the top three entries in the table point us to a slightly more complicated question: can we generalise over spelling variation? Is there any difference between *bonza* and *bonzer* (and a few other variations further down)? The answer to this may depend on what we are investigating. For example, we might want to find out whether the spelling variation is associated with age: do younger people prefer one spelling? But when we are counting the most popular responses, does it make sense to separate *bonza* and *bonzer*?

others to discuss:
- small variations: *ripper, cracker*
- *beauty* and variants
- is *you ....* different?
- cf *she.....* - are templates expressions?

The computational resource we can use to do normalisation across spelling variants is **regular expressions** or regex. Regex is a pattern matching language which allows us to find complex patterns in text data. In the code here, we only use a small part of the resources of regex:
- the pipe character | separates alternatives
- round brackets enclose set of possibilities
- \b indicates a word boundary, in the code we have ```\\b``` because the backslash is a reserved character and has to be *escaped*
Note that in some cases the regex does not specify alternatives, but because no final word boundary is included ```\\bripp``` will match any word which starts *ripp*, including *ripper* and *ripping*.

In [ ]:
## categorising - 
## add another column to the data frame

responses_postcodes$slang_grouped <- NA
for (i in 1:nrow(responses_postcodes)){
  if (str_detect(responses_postcodes[i, 'slang_lower'], 'bonz(a|er)')){
      responses_postcodes[i, 'slang_grouped'] <- 'bonza'
  } else if (str_detect(responses_postcodes[i, 'slang_lower'], '\\bbe(aut|ut|aud|w)')){
      responses_postcodes[i, 'slang_grouped'] <- 'beauty'
  } else if (str_detect(responses_postcodes[i, 'slang_lower'], '\\bgrous')) {
      responses_postcodes[i, 'slang_grouped'] <- 'grouse'
  } else if (str_detect(responses_postcodes[i, 'slang_lower'], '\\bripp')) {
      responses_postcodes[i, 'slang_grouped'] <- 'ripper'
  } else if (str_detect(responses_postcodes[i, 'slang_lower'], '\\bcrack')) {
      responses_postcodes[i, 'slang_grouped'] <- 'cracker'
  } else {
    responses_postcodes[i, 'slang_grouped'] <- responses_postcodes[i, 'slang_lower']
  }
}


## Look again
sort(table(responses_postcodes$slang_grouped), decreasing = TRUE)


If you look at the frequency table in detail, you will see that we haven't captured everything. For example, there is a *bonser* down there, and if you want to explore regex further, you could try changing the code to capture this variant as well. But there is a clear gap between the fifth most popular response and the sixth as well as clear gaps between each of the top five responses. A more exhaustive analysis of the variants might change the numbers slightly but it is very unlikely that the overall picture will change. For current purposes, this is good enough. 

**Complication**: But before we move on to getting counts by states, let's explore a little the problem of multi-word answers in this data. In some cases, respondents have provided a multi-word expression (e.g. *you little beauty*, in other cases, they have provided alternative answers (e.g. *beauty, bonza*). The regex above will ignore that difference and assigns a response to the group which is matched first from the five alternatives (if there is any match). What features of these resposnses might we be able to use to at least see how complex the problem is?

Some respondents have been helpful and used the word *or* to separate alternative answers. On the assumption that *or* is very unlikely to occur as part of a response expression, we can extract all of these examples and examine them.

**NB**: quotation marks/apostrophes are reserved characters, so they are escaped in the output which follows.

In [ ]:
## includes 'or'
responses_or <- subset(responses_postcodes, str_detect(responses_postcodes$slang_lower, '\\bor\\b'))
responses_or$slang_lower

This approach is picking up a lot of listing of responses. But there are still complications! There is one example where the *or* is used to separate two glosses (*bonza meaning good or nice*) and there are resposnes which include more than two expressions, with the position of *or* being unpredictable (*bonza beaut or beauty*, *bonza or rippa (ripper). deadly.*)

We can also make the assumption that response expressions are very unlikely to include commas or semi-colons and that responses which include those punctuation symbols include alternative responses.

In [ ]:
## includes comma or semi-colon
responses_comma <- subset(responses_postcodes, str_detect(responses_postcodes$slang_lower, ';|,'))
responses_comma$slang_lower

Again, it looks as though this approach is quite good for identifying alternative responses (but there is one example which contradicts my initial assumption about punctuation: *winner, winner chicken dinner*).

Using both *or* and punctuation, we could make a function which would parse out the alternative responses and add new lines to the dataframe consisting of a single response item and a copy of all the other fields associated with the original response. We can see that this would not be completely accurate and this raises a question which often arises in cleaning messy data: how much effort is it worth expending to catch one or two anomalies?

Approaching the problem from the opposite direction, we can make the assumption that multi-word responses which do **not** include *or* or relevant punctuation do consist of multi-word expressions.

In [ ]:
## includes space but not 'or', comma, semi-colon
response_multi <- subset(responses_postcodes, str_detect(responses_postcodes$slang_lower, ' ') & !str_detect(responses_postcodes$slang_lower, ';|,|or'))
response_multi$slang_lower

Again, this approach seems to work quite well, but also again, there are anomalies. Is *bonzer ripper grouse* a single expression, or did the respondent just not bother to indicate that these are alternatives? There are examples which are a single word plus some explanation, e.g. *beaudy (beauty with a \'d\' replacing the \'t\')*. And there are examples with dividers between alternatives which we haven't thought about: *beauty/ ripper*. Which all brings us back to the question raised previously: how much effort is it worth expending to catch these anomalies?

More positively, I hope the preceding discussion has shown that making plausible (sensible) assumptions and using simple computational approaches based on those assumptions can at least give us a good picture of complex, messy data.

Moving on, now we can get counts by state for the most popular responses. We do this by extracting subsets of responses depending on the postcodes and therefore we need to start by setting up a dataframe which has the ranges of postcodes for the different states (I'm treating ACT as part of NSW here....). 

In [ ]:
## get counts by state/territory

## set up post code ranges, output table
states <- c('NSW', 'VIC', 'QLD', 'SA', 'WA', 'TAS', 'NT')
minima <- c(2000, 3000, 4000, 5000, 6000, 7000, 0800)
maxima <- c(2999, 3999, 4999, 5999, 6999, 7999, 0899)
pc_ranges <- data.frame(states, minima, maxima)

top5 <- data.frame(slang = c('bonza', 'beauty', 'grouse', 'ripper', 'cracker'))
top5[, c('NSW', 'VIC', 'QLD', 'SA', 'WA', 'TAS', 'NT')] = NA

In [ ]:
for (i in 1:length(states)) {
    state_responses <- responses_postcodes[responses_postcodes$postCode >= pc_ranges[i, 2] & responses_postcodes$postCode <= pc_ranges[i, 3],]
    
    for (j in 1:nrow(top5)) {
        slang <- state_responses[state_responses$slang_grouped == top5[j, 1], ]
        top5[j, i + 1] <- nrow(slang)
        }
        top5[6, i+1] <- (nrow(state_responses))
    }
top5[6,1] <- 'Total'
top5

Remember that the data format we want has each geographic entity (here, the states) as a row in the table. So we need to transpose the ```top5``` table. We also add a column showing the percentage of responses for each state for each of the five responses. We do this because comparing raw counts won't tell us much except that there are more responses from the more populous states.

In [ ]:
top5_byState <- as.data.frame(t(top5))
colnames(top5_byState) <- top5_byState[1,]
top5_byState <- top5_byState[-c(1),]
top5_byState[,1:6] <- as.numeric(unlist(top5_byState[,1:6]))

for (i in 1:5) {
    top5_byState[, i+6] <- (top5_byState[,i] * 100)/top5_byState[, 6]
    }

colnames(top5_byState) <- c('bonza_raw', 'beauty_raw', 'grouse_raw', 'ripper_raw', 'cracker_raw', 'Total', 'bonza_percent', 'beauty_percent', 'grouse_percent', 'ripper_percent', 'cracker_percent')
top5_byState                 

Now we can put together the table which will be the input for making maps. The ```ozmaps``` package provides a list of polygon data for the states of Australia. We discard ACT (see above) and Other Territories and then add our data about the top five responses by state as additional columns in the table.

In [ ]:
## putting mapping data together

slang_map <- ozmap_data('states')
## discard ACT and Other Territories
slang_map <- slang_map[-c(8,9), ]
slang_map <- cbind(slang_map, top5_byState)

In [ ]:
str(slang_map)

I want to make an array of maps so that it is easy to compare the geographic distribution of the different responses. To do this, rather than immediately displaying the maps, I assign them to plot objects which can then be used by the ```patchwork``` package to make the nice array. (```p6``` is an empty plot used to make all the other plots equal in size.)

In [ ]:
## create plot objects including a blank plot

for (i in 1:5){
    col_name = paste(top5[i, 'slang'], '_percent', sep = '')
    legend = paste('% responses\n"something very good" = "', top5[i, 'slang'], '"', sep = '')
    plot <- ggplot(slang_map) + geom_sf(aes(fill = .data[[col_name]])) + scale_color_identity() + scale_fill_viridis(option = 'F', begin = 1, end = 0.4, limits=c(0, 60)) + labs(fill = legend)
    assign(paste0("p", i, sep=''), plot)
    }
p6 <- ggplot() + theme(panel.background = element_rect(fill = "white", colour = "white"), plot.background = element_rect(fill = "white", colour = "white"))


In [ ]:
## use patchwork to create array of plots

(p1 + p2 + p3) / (p4 + p5 + p6)